In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

* # Yak: Week 5 benchmark on unseen data

Uses the **supervised checkpoint `best.pt`** from `Yak_Finetune_Mixture.ipynb`. No training happens here and `Data/processed` is not needed.
The RLCD run did not meet its criteria (Brier and ECE were not better than the supervised model with temperature scaling), so
the supervised model is the main subject. If `rlcd_seed0.pt` exists it is scored next to it for reference.

```
 best.pt  ->  [candidate labels = the dataset's class names, 2 wordings each]
              [4 label orders averaged]
              -> accuracy, macro F1, Brier, NLL, ECE   (zero-shot: no benchmark label was used)
 compared with: chance, majority class, TF-IDF + logistic regression (supervised, 1,000 rows), a diagnostic temperature
```

Datasets: **AG News** (4 topics), **Emotion** (6 emotions), **SST-2** (2 sentiments). The model never saw these texts or these label names.

How to read the results:
- Accuracy is shown with a 95% interval (`+-`) from the number of test rows, so small gaps are not over-read.
- SST-2 has two classes (chance 50%), so it is the least informative; look at AG News and Emotion first.
- Two label wordings per dataset show how much the model depends on how a class is named.
- The "diagnostic temperature" is fitted on 1,000 benchmark training rows. It is not zero-shot; it shows how much of the calibration error is only a scale problem.

In [ ]:
import os, sys, json, random, time, types, subprocess, copy

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                   # Google Colab: mount Drive and get the repository
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/Yak"):
        subprocess.run(["git", "clone", "--branch", "feature/finetune",
                        "https://github.com/gowthambalachandhiran/Yak.git", "/content/Yak"], check=True)
    os.chdir("/content/Yak")

def find_root(start="."):
    p = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(p, "Model")) and os.path.isdir(os.path.join(p, "Tokenizer")):
            return p
        if os.path.dirname(p) == p:
            raise RuntimeError("run this notebook inside the Yak repository (Model/ and Tokenizer/ not found)")
        p = os.path.dirname(p)

ROOT = find_root()
os.chdir(ROOT); sys.path.insert(0, ROOT)
FT_DIR = None                                  # the folder with Utils/ (spelled FineTuning in git, Finetuning on some disks)
for cand in ("FineTuning", "Finetuning"):
    if os.path.isdir(os.path.join(ROOT, "Model", "OurExperiments", cand, "Utils")):
        FT_DIR = os.path.join(ROOT, "Model", "OurExperiments", cand)
        break
assert FT_DIR, "Model/OurExperiments/FineTuning/Utils not found"
sys.path.insert(0, FT_DIR)
print("repository root:", ROOT)
print("helper modules :", os.path.join(FT_DIR, "Utils"))

In [ ]:
# ---------------- configuration (edit these) ----------------
CKPT_CANDIDATES = [                                       # the supervised best.pt: the first existing path is used
    os.environ.get("YAK_CKPT", ""),
    "/content/drive/MyDrive/Yak/finetune_mixture/best.pt",
    "runs/mixture/best.pt",
    "best.pt",
]
RLCD_CKPT = "runs/rlcd/rlcd_seed0.pt"                     # optional, scored for reference if the file exists
OUT_DIR = "/content/drive/MyDrive/Yak/benchmark" if IN_COLAB else "runs/benchmark"

BENCH = ["ag_news", "emotion", "sst2"]                    # downloaded with the `datasets` library (pip install datasets)
BENCH_LIMIT = 2000                                        # test rows per dataset
BENCH_FIT = 1000                                          # benchmark train rows: ONLY for the diagnostic temperature and the TF-IDF reference
TTA = 4                                                   # label orders averaged
EVAL_BATCH = 64
SEED = 0
D_MODEL, NHEAD, LAYERS, MAX_SEQ, DROPOUT = 128, 4, 4, 2048, 0.1   # must match the checkpoint

CKPT = next((p for p in CKPT_CANDIDATES if p and os.path.exists(p)), None)
if CKPT is None:
    raise FileNotFoundError("best.pt not found. Tried:\n  " + "\n  ".join(p or "(YAK_CKPT not set)" for p in CKPT_CANDIDATES))
os.makedirs(OUT_DIR, exist_ok=True)
print("checkpoint:", CKPT)

## Packing, tokenizer and model
The packing cell, classifier and evaluation helpers are the same code as in the fine-tuning notebook.

In [ ]:
# cell: packing
import random
from Utils.FastBpe import normalize_text, normalize_label


LABEL_ALIASES = {"oos": "out of scope"}   # names that carry no meaning as words


class Packer:
    def __init__(self, fast_bpe, class_names, max_labels=160, max_text_tokens=96,
                 max_label_tokens=12, aliases=None, max_premise_tokens=64, max_hypothesis_tokens=32):
        """
        fast_bpe:        FastBPE
        class_names:     list of class names, the position is the class id
        max_labels:      upper bound M on labels in one packed example
        max_text_tokens: the text is cut to this many tokens
        max_label_tokens: each label is cut to this many tokens
        """
        self.bpe = fast_bpe
        self.class_names = list(class_names)
        self.num_classes = len(self.class_names)
        self.max_labels = min(max_labels, self.num_classes)
        self.train_ids = list(range(self.num_classes))     # classes that may appear as training candidates
        self.max_text_tokens = max_text_tokens
        self.max_premise_tokens = max_premise_tokens
        self.max_hypothesis_tokens = max_hypothesis_tokens
        aliases = LABEL_ALIASES if aliases is None else aliases
        self.label_ids = []
        for name in self.class_names:
            ids = fast_bpe.encode_ids(normalize_label(aliases.get(name, name)), normalize=True)
            ids = ids[:max_label_tokens] or [fast_bpe.unk_id]
            self.label_ids.append(ids)

    # ------------------------------------------------------------------ text
    def encode_text(self, text):
        return self.bpe.encode_ids(text, normalize=True)[: self.max_text_tokens]

    def encode_pair(self, premise, hypothesis):
        """Sentence pair as one 'text': premise tokens [SEP] hypothesis tokens, each cut separately.
        The model has no segment embeddings, so the [SEP] between them is the only divider."""
        p = self.bpe.encode_ids(premise, normalize=True)[: self.max_premise_tokens]
        h = self.bpe.encode_ids(hypothesis, normalize=True)[: self.max_hypothesis_tokens]
        return p + [self.bpe.sep_id] + h

    def encode_row(self, row):
        if "premise" in row:
            return self.encode_pair(row["premise"], row["hypothesis"])
        return self.encode_text(row["text"])

    def max_text_len(self):
        return max(self.max_text_tokens, self.max_premise_tokens + 1 + self.max_hypothesis_tokens)

    # ------------------------------------------------------------------ packing
    def pack(self, text_ids, class_order):
        """Return (ids, label_pos) for the given text ids and classes in the given order."""
        ids = [self.bpe.cls_id] + list(text_ids) + [self.bpe.sep_id]
        label_pos = []
        for c in class_order:
            label_pos.append(len(ids))
            ids.extend(self.label_ids[c])
            ids.append(self.bpe.sep_id)
        return ids, label_pos

    def max_packed_len(self):
        longest = sorted((len(x) + 1 for x in self.label_ids), reverse=True)[: self.max_labels]
        return 2 + self.max_text_len() + sum(longest)

    def set_train_classes(self, class_ids):
        """Restrict training candidates to class_ids (the rest are held out and never shown in training).
        Evaluation still uses every class."""
        self.train_ids = sorted(class_ids)

    # ------------------------------------------------------------------ choosing the label list
    def train_labels(self, true_class, rng, full_prob=0.5, min_labels=2):
        """Random label list that always contains true_class.

        With probability full_prob use as many labels as allowed (so the model sees the real test
        condition), otherwise draw the count uniformly from min_labels..max_labels.
        """
        assert true_class in self.train_ids, f"class {true_class} is held out and cannot be a training target"
        cap = min(self.max_labels, len(self.train_ids))
        lo = min(min_labels, cap)
        n = cap if rng.random() < full_prob else rng.randint(lo, cap)
        others = [c for c in self.train_ids if c != true_class]
        chosen = rng.sample(others, n - 1) + [true_class]
        rng.shuffle(chosen)
        return chosen

    def eval_labels(self, true_class, rng=None):
        """All classes (up to max_labels). If there are more classes than max_labels, keep the true
        label plus a fixed-seed random subset. Order is class-id order unless rng shuffles it."""
        order = list(range(self.num_classes))
        if self.num_classes > self.max_labels:
            sub_rng = random.Random(true_class)
            others = [c for c in order if c != true_class]
            order = sorted(sub_rng.sample(others, self.max_labels - 1) + [true_class])
        if rng is not None:
            rng.shuffle(order)
        return order


class ClsDataset:
    """Map-style dataset (works with torch DataLoader, no torch import needed here).

    rows:   list of dicts with "text" and "label" (the jsonl format written by Data/*.py)
    train:  True  -> random label subsets and order, different every epoch (call set_epoch)
            False -> all labels in class order
    """

    def __init__(self, rows, packer, train, seed=0, full_prob=0.5, min_labels=2, eval_shuffle_seed=None, task_id=0):
        self.packer = packer
        self.task_id = task_id
        self.train = train
        self.seed = seed
        self.epoch = 0
        self.full_prob = full_prob
        self.min_labels = min_labels
        self.eval_shuffle_seed = eval_shuffle_seed   # None = class-id order; int = a fixed shuffled order
        self.text_ids = [packer.encode_row(r) for r in rows]      # single texts or premise/hypothesis pairs
        self.labels = [int(r["label"]) for r in rows]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.example(idx, self.epoch * 10_007 + idx)

    def example(self, idx, nonce):
        """Example idx with a label list drawn from `nonce` (a different nonce gives a different list)."""
        true_class = self.labels[idx]
        if self.train:
            rng = random.Random(self.seed * 1_000_003 + nonce)
            order = self.packer.train_labels(true_class, rng, self.full_prob, self.min_labels)
        else:
            rng = None
            if self.eval_shuffle_seed is not None:
                rng = random.Random(self.eval_shuffle_seed * 1_000_003 + idx)
            order = self.packer.eval_labels(true_class, rng)
        ids, label_pos = self.packer.pack(self.text_ids[idx], order)
        return {"ids": ids, "label_pos": label_pos, "slot_class": order,
                "target": order.index(true_class), "true_class": true_class, "task_id": self.task_id}


def collate_lists(batch, pad_id):
    """Pad a list of examples to the longest sequence and the largest label count in the batch.

    Returns plain python lists so it can be tested without torch:
        ids [B, L] (pad_id), label_pos [B, M] (0 for unused), label_mask [B, M] (True = real label),
        slot_class [B, M] (-1 for unused), target [B], true_class [B]
    """
    L = max(len(b["ids"]) for b in batch)
    M = max(len(b["label_pos"]) for b in batch)
    out = {"ids": [], "label_pos": [], "label_mask": [], "slot_class": [], "target": [], "true_class": [],
           "task_id": []}
    for b in batch:
        n = len(b["label_pos"])
        out["ids"].append(b["ids"] + [pad_id] * (L - len(b["ids"])))
        out["label_pos"].append(b["label_pos"] + [0] * (M - n))
        out["label_mask"].append([True] * n + [False] * (M - n))
        out["slot_class"].append(b["slot_class"] + [-1] * (M - n))
        out["target"].append(b["target"])
        out["true_class"].append(b["true_class"])
        out["task_id"].append(b.get("task_id", 0))
    return out


def make_collate(pad_id):
    """Collate function for torch DataLoader: python lists -> tensors."""
    def collate(batch):
        import torch
        d = collate_lists(batch, pad_id)
        return {
            "ids": torch.tensor(d["ids"], dtype=torch.long),
            "label_pos": torch.tensor(d["label_pos"], dtype=torch.long),
            "label_mask": torch.tensor(d["label_mask"], dtype=torch.bool),
            "slot_class": torch.tensor(d["slot_class"], dtype=torch.long),
            "target": torch.tensor(d["target"], dtype=torch.long),
            "true_class": torch.tensor(d["true_class"], dtype=torch.long),
            "task_id": torch.tensor(d["task_id"], dtype=torch.long),
        }
    return collate


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from Tokenizer.Tokenizer import BPE
from Utils.FastBpe import FastBPE

bpe = BPE(); bpe.load("Tokenizer/TokenFile/bpe.json")
fb = FastBPE(bpe)
print("tokenizer ready, vocabulary", fb.vocab_size)

In [ ]:
# cell: classifier
import torch
import torch.nn as nn
import torch.nn.functional as F

from Model.LayerOne.EmbedPostional import EmbedPos
from Model.EncoderBlock import EncoderBlock


class YakClassifier(nn.Module):
    """Label-scoring classifier on top of the pre-trained Yak encoder.

    packed ids -> EmbedPos -> N x EncoderBlock -> final LayerNorm -> h [B, L, d]
    gather h at the first token of every label  -> H [B, M, d]
    shared scorer Linear(d, 1)                  -> scores [B, M]
    padded slots set to -inf                    -> softmax over slots happens in the loss / at inference

    The layer names (embed, blocks, final_norm) match MlmModel, so the phase 1 checkpoint loads directly.
    The MLM head and its bias are not part of this model.
    """

    def __init__(self, vocab_size: int, d_model: int, nhead: int, num_layers: int,
                 max_seq_len: int, dropout: float = 0.1, pad_token_id: int = 0):
        super().__init__()
        self.pad_id = pad_token_id
        self.embed = EmbedPos(vocab_size, d_model=d_model, max_len=max_seq_len,
                              pad_id=pad_token_id, dropout=dropout)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, nhead, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.scorer = nn.Linear(d_model, 1)           # the only new weights: d_model + 1
        nn.init.normal_(self.scorer.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.scorer.bias)

    # ------------------------------------------------------------------ weights
    def load_pretrained(self, path: str, map_location="cpu"):
        """Load embedding, blocks and final norm from a phase 1 checkpoint. Returns a short report."""
        try:
            sd = torch.load(path, map_location=map_location, weights_only=True)
        except Exception:                                   # checkpoint holds more than plain tensors
            sd = torch.load(path, map_location=map_location, weights_only=False)   # only for your own files
        meta = {k: v for k, v in sd.items() if isinstance(v, (int, float, str))} if isinstance(sd, dict) else {}
        if isinstance(sd, dict):
            for key in ("model", "model_state", "model_state_dict", "state_dict"):
                if key in sd and isinstance(sd[key], dict):
                    sd = sd[key]
                    break
        clean = {}
        for k, v in sd.items():
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
            if k.startswith(("embed.", "blocks.", "final_norm.")):
                clean[k] = v
        missing, unexpected = self.load_state_dict(clean, strict=False)
        missing = [k for k in missing if not k.startswith("scorer.")]   # scorer is new, expected missing
        if missing or unexpected or not clean:
            raise RuntimeError(f"checkpoint does not match: missing={missing[:5]} unexpected={unexpected[:5]} "
                               f"loaded={len(clean)}")
        return {"loaded_tensors": len(clean), "head_keys_ignored": [k for k in sd if k.startswith("head")],
                "checkpoint_info": meta}

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def set_encoder_trainable(self, trainable: bool):
        """Freeze or unfreeze embedding and blocks. The final norm and scorer always train."""
        for p in self.embed.parameters():
            p.requires_grad = trainable
        for p in self.blocks.parameters():
            p.requires_grad = trainable

    # ------------------------------------------------------------------ forward
    def encode(self, ids: torch.Tensor) -> torch.Tensor:
        keep = ids != self.pad_id
        padding_mask = None if bool(keep.all()) else keep
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, padding_mask)
        return self.final_norm(x)

    def score(self, ids: torch.Tensor, label_pos: torch.Tensor, label_mask: torch.Tensor) -> torch.Tensor:
        """ids [B, L], label_pos [B, M] (0 for unused slots), label_mask [B, M] True for real labels.
        Returns scores [B, M] with -inf in unused slots."""
        h = self.encode(ids)                                                   # [B, L, d]
        idx = label_pos.unsqueeze(-1).expand(-1, -1, h.size(-1))               # [B, M, d]
        H = torch.gather(h, 1, idx)                                            # [B, M, d]
        scores = self.scorer(H).squeeze(-1)                                    # [B, M]
        return scores.masked_fill(~label_mask, float("-inf"))

    def forward(self, ids, label_pos, label_mask, target=None, loss_type: str = "ce"):
        """target is the slot index of the true label, shape [B]. Returns scores, or (loss, scores)."""
        scores = self.score(ids, label_pos, label_mask)
        if target is None:
            return scores
        return self.loss(scores, target, loss_type), scores

    @staticmethod
    def loss(scores: torch.Tensor, target: torch.Tensor, loss_type: str = "ce") -> torch.Tensor:
        scores = scores.float()
        if loss_type == "ce":
            return F.cross_entropy(scores, target)
        if loss_type == "brier":
            p = F.softmax(scores, dim=-1)                                      # -inf slots become exactly 0
            y = F.one_hot(target, p.size(-1)).to(p.dtype)
            return ((p - y) ** 2).sum(dim=-1).mean()
        raise ValueError(f"unknown loss {loss_type}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)      # your own file
    assert isinstance(ck, dict) and "model" in ck, f"{path}: expected a checkpoint with a 'model' key"
    m = YakClassifier(fb.vocab_size, D_MODEL, NHEAD, LAYERS, MAX_SEQ, DROPOUT, fb.pad_id).to(device)
    m.load_state_dict(ck["model"])                                     # strict: every tensor must match
    m.eval()
    return m, ck

model, ck = load_model(CKPT)
models = {"SFT (best.pt)": model}
print(f"device {device} | parameters {model.num_parameters():,} | checkpoint epoch {ck.get('epoch')} | tasks {ck.get('tasks')}")
if RLCD_CKPT and os.path.exists(RLCD_CKPT):
    models["RLCD (reference)"] = load_model(RLCD_CKPT)[0]
    print("also loaded", RLCD_CKPT)

In [ ]:
# cell: eval_helpers
import contextlib
import numpy as np
import torch
from torch.utils.data import DataLoader
from Utils.Metrics import softmax, slots_to_class_probs

def amp_setup(device, use_amp):
    if use_amp and device.type == "cuda":
        # bf16 needs an Ampere (compute capability 8+) GPU; a T4 is 7.5 and should use fp16 (plus loss scaling)
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        return (lambda: torch.autocast("cuda", dtype=dtype)), dtype
    return (lambda: contextlib.nullcontext()), None


@torch.no_grad()
def predict(model, dataset, device, batch_size, collate, autocast):
    """Return slot scores [N, M], slot classes [N, M] (-1 = unused), target slots [N], true classes [N]."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    S, C, T, Y = [], [], [], []
    for b in loader:
        with autocast():
            scores = model(b["ids"].to(device), b["label_pos"].to(device), b["label_mask"].to(device))
        S.append(scores.float().cpu().numpy())
        C.append(b["slot_class"].numpy())
        T.append(b["target"].numpy())
        Y.append(b["true_class"].numpy())
    M = max(s.shape[1] for s in S)
    S = np.concatenate([np.pad(s, ((0, 0), (0, M - s.shape[1])), constant_values=-np.inf) for s in S])
    C = np.concatenate([np.pad(c, ((0, 0), (0, M - c.shape[1])), constant_values=-1) for c in C])
    return S, C, np.concatenate(T), np.concatenate(Y)


def eval_probs(model, rows, packer, args, device, collate, autocast, n_orders=1, temperature=1.0):
    """Class probabilities [N, K] averaged over n_orders label orders (the first is class-id order)."""
    acc, true, first = None, None, None
    for k in range(n_orders):
        ds = ClsDataset(rows, packer, train=False, eval_shuffle_seed=None if k == 0 else k)
        S, C, T, Y = predict(model, ds, device, args.eval_batch_size, collate, autocast)
        if k == 0:
            first = (S, T)
        P = slots_to_class_probs(softmax(S, temperature), C, packer.num_classes)
        acc = P if acc is None else acc + P
        true = Y
    return acc / n_orders, true, first


In [ ]:

import types
from torch.utils.data import DataLoader
from Utils.Metrics import report, fit_temperature, softmax, slots_to_class_probs

ev_args = types.SimpleNamespace(eval_batch_size=EVAL_BATCH)
autocast, amp_dtype = amp_setup(device, True)
collate = make_collate(fb.pad_id)

def macro_f1(P, Y, K):
    pred, f = P.argmax(1), []
    for c in range(K):
        tp = int(((pred == c) & (Y == c)).sum()); fp = int(((pred == c) & (Y != c)).sum()); fn = int(((pred != c) & (Y == c)).sum())
        if tp + fp + fn:
            f.append(2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f)) if f else 0.0

def rel_curve(P, Y, bins=10, min_n=15):
    conf, hit = P.max(1), (P.argmax(1) == Y)
    edges, xs, ys = np.linspace(0, 1, bins + 1), [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= min_n:
            xs.append(float(conf[m].mean())); ys.append(float(hit[m].mean()))
    return xs, ys

def ci95(acc, n):
    return 1.96 * (acc * (1 - acc) / n) ** 0.5

## Benchmark data
Class names are the candidate labels. Two wordings per dataset show sensitivity to naming.

In [ ]:
BENCH_CFG = {
    "ag_news": dict(hf=("ag_news",), text="text", train="train", test="test",
                    variants={"plain": ["world", "sports", "business", "science and technology"],
                              "news": ["world news", "sports news", "business news", "science and technology news"]}),
    "emotion": dict(hf=("dair-ai/emotion",), text="text", train="train", test="test",
                    variants={"plain": ["sadness", "joy", "love", "anger", "fear", "surprise"],
                              "adjective": ["sad", "happy", "loving", "angry", "scared", "surprised"]}),
    "sst2":    dict(hf=("stanfordnlp/sst2",), text="sentence", train="train", test="validation",
                    variants={"plain": ["negative", "positive"], "goodbad": ["bad", "good"]}),
}

def load_bench(name):
    from datasets import load_dataset                         # pip install datasets
    cfg = BENCH_CFG[name]
    ds = load_dataset(*cfg["hf"])
    r = random.Random(SEED)
    def rows(split, n):
        idx = list(range(len(ds[split]))); r.shuffle(idx)
        out = []
        for i in idx[:n]:
            row = ds[split][i]
            out.append({"text": row[cfg["text"]], "label": int(row["label"])})
        return out
    return {"name": name, "test": rows(cfg["test"], BENCH_LIMIT), "fit": rows(cfg["train"], BENCH_FIT), "cfg": cfg}

bench = {}
for name in BENCH:
    try:
        bench[name] = load_bench(name)
        b = bench[name]; K = len(b["cfg"]["variants"]["plain"])
        counts = np.bincount([x["label"] for x in b["test"]], minlength=K)
        print(f"{name:9s} test {len(b['test'])}  fit {len(b['fit'])}  classes {K}  majority class {counts.max() / counts.sum():.3f}  chance {1 / K:.3f}")
        print("   example:", b["test"][0]["text"][:100].replace("\n", " "), "->", b["cfg"]["variants"]["plain"][b["test"][0]["label"]])
    except Exception as e:
        print(f"skipping {name}: {type(e).__name__}: {str(e)[:120]}  (pip install datasets, and check network access)")
assert bench, "no benchmark dataset could be loaded"

## Scoring

In [ ]:
def bench_eval(m, name, variant, temp=1.0, split="test"):
    b = bench[name]
    labels = b["cfg"]["variants"][variant]
    pk = Packer(fb, labels, max_labels=len(labels), aliases={})
    P, Y, first = eval_probs(m, b[split], pk, ev_args, device, collate, autocast, TTA, temp)
    met = report(P, Y); met["f1"] = macro_f1(P, Y, len(labels)); met["n"] = len(Y)
    return met, P, Y, first

def fit_bench_T(m, name):
    b = bench[name]
    labels = b["cfg"]["variants"]["plain"]
    pk = Packer(fb, labels, max_labels=len(labels), aliases={})
    _, _, first = eval_probs(m, b["fit"], pk, ev_args, device, collate, autocast, 1, 1.0)
    return fit_temperature(first[0], first[1])

main = "SFT (best.pt)"
bres, bstore = {}, {}
for name in bench:
    for variant in bench[name]["cfg"]["variants"]:
        for mname, m in models.items():
            met, P, Y, _ = bench_eval(m, name, variant)
            bres[(name, variant, mname, "T=1")] = met; bstore[(name, variant, mname, "T=1")] = (P, Y)
        print(f"scored {name} / {variant}")
    bT = fit_bench_T(model, name)
    met, P, Y, _ = bench_eval(model, name, "plain", bT)
    bres[(name, "plain", main, "T diag")] = met; bstore[(name, "plain", main, "T diag")] = (P, Y)
    bres[(name, "plain", main, "T value")] = {"T": bT}

tfidf = {}
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    for name, b in bench.items():
        K = len(b["cfg"]["variants"]["plain"])
        vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        Xf = vec.fit_transform([x["text"] for x in b["fit"]]); yf = [x["label"] for x in b["fit"]]
        clf = LogisticRegression(max_iter=1000, C=5.0).fit(Xf, yf)
        P = np.zeros((len(b["test"]), K)); P[:, clf.classes_] = clf.predict_proba(vec.transform([x["text"] for x in b["test"]]))
        Y = np.array([x["label"] for x in b["test"]])
        met = report(P, Y); met["f1"] = macro_f1(P, Y, K); met["n"] = len(Y)
        tfidf[name] = (met, P, Y)
except Exception as e:
    print("TF-IDF reference skipped:", type(e).__name__, str(e)[:100], "(pip install scikit-learn)")

cols = ("acc", "f1", "brier", "nll", "ece")
for name, b in bench.items():
    K = len(b["cfg"]["variants"]["plain"])
    counts = np.bincount([x["label"] for x in b["test"]], minlength=K)
    print(f"\n{name}  (n={len(b['test'])}, {K} classes, {TTA} label orders)")
    print(f"  {'model':36s}" + "".join(f"{c:>9s}" for c in cols) + "   acc 95% CI")
    print(f"  {'chance':36s}{1 / K:9.3f}")
    print(f"  {'majority class':36s}{counts.max() / counts.sum():9.3f}")
    for mname in models:
        for variant in b["cfg"]["variants"]:
            r_ = bres[(name, variant, mname, "T=1")]
            print(f"  {mname + ' [' + variant + ']':36s}" + "".join(f"{r_[c]:9.3f}" for c in cols) + f"   +-{ci95(r_['acc'], r_['n']):.3f}")
    r_ = bres[(name, "plain", main, "T diag")]
    tv = bres[(name, "plain", main, "T value")]["T"]
    print(f"  {'SFT + diagnostic T=' + format(tv, '.2f'):36s}" + "".join(f"{r_[c]:9.3f}" for c in cols))
    if name in tfidf:
        print(f"  {'TF-IDF + LR (supervised, ' + str(BENCH_FIT) + ' rows)':36s}" + "".join(f"{tfidf[name][0][c]:9.3f}" for c in cols))

## Plots

In [ ]:
names_b = list(bench)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
x = np.arange(len(names_b)); w = 0.25
for k, (metric, title) in enumerate((("acc", "accuracy (higher is better)"), ("brier", "Brier (lower is better)"), ("ece", "ECE (lower is better)"))):
    ax[k].bar(x - w, [bres[(n, "plain", main, "T=1")][metric] for n in names_b], w, label="SFT")
    ax[k].bar(x, [bres[(n, "plain", main, "T diag")][metric] for n in names_b], w, label="SFT + diagnostic T")
    if metric == "acc":
        ax[k].scatter(x, [1 / len(bench[n]["cfg"]["variants"]["plain"]) for n in names_b], color="k", marker="_", s=700, label="chance")
        if tfidf:
            ax[k].bar(x + w, [tfidf[n][0]["acc"] if n in tfidf else 0 for n in names_b], w, color="#bbbbbb", label="TF-IDF (supervised)")
        ax[k].errorbar(x - w, [bres[(n, "plain", main, "T=1")]["acc"] for n in names_b],
                       yerr=[ci95(bres[(n, "plain", main, "T=1")]["acc"], bres[(n, "plain", main, "T=1")]["n"]) for n in names_b], fmt="none", color="k", capsize=3)
    ax[k].set_xticks(x); ax[k].set_xticklabels(names_b); ax[k].set_title(title)
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, len(names_b), figsize=(5 * len(names_b), 4), squeeze=False)
for j, n in enumerate(names_b):
    for label, key, style in (("SFT", (n, "plain", main, "T=1"), "--"), ("SFT + diagnostic T", (n, "plain", main, "T diag"), "-")):
        xs, ys = rel_curve(*bstore[key])
        ax[0, j].plot(xs, ys, style, marker="o", label=label)
    ax[0, j].plot([0, 1], [0, 1], color="gray", ls=":"); ax[0, j].set_title(f"{n}: reliability (unseen data)")
    ax[0, j].set_xlabel("confidence"); ax[0, j].set_ylabel("accuracy"); ax[0, j].legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, len(names_b), figsize=(4.5 * len(names_b), 3.2), squeeze=False)
for j, n in enumerate(names_b):
    variants = list(bench[n]["cfg"]["variants"]); xv = np.arange(len(variants))
    ax[0, j].bar(xv, [bres[(n, v, main, "T=1")]["acc"] for v in variants], 0.5)
    ax[0, j].axhline(1 / len(bench[n]["cfg"]["variants"]["plain"]), color="gray", ls=":", label="chance")
    ax[0, j].set_xticks(xv); ax[0, j].set_xticklabels([f"{v}\n" + ", ".join(bench[n]["cfg"]["variants"][v][:2]) + ".." for v in variants], fontsize=7)
    ax[0, j].set_title(f"{n}: accuracy by label wording"); ax[0, j].legend(fontsize=8)
plt.tight_layout(); plt.show()

# confusion matrix for the plain wording (rows = true, columns = predicted)
fig, ax = plt.subplots(1, len(names_b), figsize=(4.5 * len(names_b), 4), squeeze=False)
for j, n in enumerate(names_b):
    P, Y = bstore[(n, "plain", main, "T=1")]; K = P.shape[1]
    cm = np.zeros((K, K)); np.add.at(cm, (Y, P.argmax(1)), 1); cm = cm / cm.sum(1, keepdims=True)
    ax[0, j].imshow(cm, vmin=0, vmax=1, cmap="Blues"); ax[0, j].set_title(f"{n}: confusion (row-normalised)")
    names_ = bench[n]["cfg"]["variants"]["plain"]
    ax[0, j].set_xticks(range(K)); ax[0, j].set_yticks(range(K)); ax[0, j].set_xticklabels(names_, rotation=60, ha="right", fontsize=7); ax[0, j].set_yticklabels(names_, fontsize=7)
    for a in range(K):
        for b_ in range(K):
            ax[0, j].text(b_, a, f"{cm[a, b_]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm[a, b_] > 0.5 else "black")
plt.tight_layout(); plt.show()

In [ ]:
from Utils.Infer import classify
rs = random.Random(1)
for name, b in bench.items():
    labels = b["cfg"]["variants"]["plain"]
    for row in rs.sample(b["test"], 2):
        out = classify(model, fb, row["text"], labels, Packer, device=device, n_orders=TTA)
        print(f"[{name}] true: {labels[row['label']]}\n  {row['text'][:110]}")
        print("   SFT:", [(l, round(p, 2)) for l, p in out][:3], "\n")

In [ ]:
def jsonable(d):
    return {"|".join(map(str, k)): v for k, v in d.items()}
results = {"checkpoint": CKPT, "tta": TTA, "bench_limit": BENCH_LIMIT, "bench_fit": BENCH_FIT,
           "benchmark": jsonable(bres), "tfidf": {n: v[0] for n, v in tfidf.items()}}
json.dump(results, open(os.path.join(OUT_DIR, "results_benchmark.json"), "w"), indent=2, default=float)
print("saved", os.path.join(OUT_DIR, "results_benchmark.json"))

## Reading the outcome
- **Above chance on AG News and Emotion, with intervals that do not overlap chance:** label words carry over to new topics. How far above decides how strong the claim is.
- **Close to chance:** the model memorised its training labels or the datasets are too far from banking and NLI text. Report it as such.
- **Label wording gap:** a large gap between the two wordings means the model leans on how a class is named.
- **Calibration:** compare Brier and ECE with the diagnostic-temperature row. A big improvement from one temperature means the model is mostly over- or under-confident on new data, which is a scale problem.
- **TF-IDF reference:** it saw 1,000 labelled rows, the model saw none. Matching it with zero labels is a strong result; falling well short is expected.